In [0]:
from pyspark.sql.functions import *
bronze_base = "abfss://bronze@travelappprojectstorage.dfs.core.windows.net"
silver_base = "abfss://silver@travelappprojectstorage.dfs.core.windows.net"
quarantine_base = "abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/geofence"
checkpoint_base = "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/geofence"


In [0]:

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        f"{bronze_base}/_schemas/geofence_alert"
    )
    .load(f"{bronze_base}/geofence_alert")
    .withWatermark("CreatedAt", "1 day")
    .dropDuplicates(["AlertId"])
)


In [0]:
invalid_alertid_df = stream_df.filter(col("AlertId").isNull())

invalid_tourist_df = stream_df.filter(col("TouristId").isNull())

invalid_place_df = stream_df.filter(col("PlaceId").isNull())

invalid_timestamp_df = stream_df.filter(col("CreatedAt").isNull())

invalid_distance_df = stream_df.filter(col("DistanceMeters") < 0)

invalid_severity_df = stream_df.filter(
    ~upper(col("Severity")).isin("LOW", "MEDIUM", "HIGH", "CRITICAL")
)


In [0]:

invalid_alertid_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_alertid") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_alertid")

invalid_tourist_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_tourist") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_tourist")

invalid_place_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_place") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_place")

invalid_timestamp_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_timestamp") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_timestamp")

invalid_distance_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_distance") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_distance")

invalid_severity_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base}/invalid_severity") \
    .trigger(availableNow=True) \
    .start(f"{quarantine_base}/invalid_severity")


In [0]:
clean_df = (
    stream_df.filter(
        col("AlertId").isNotNull() &
        col("TouristId").isNotNull() &
        col("PlaceId").isNotNull() &
        col("CreatedAt").isNotNull() &
        (col("DistanceMeters") >= 0) &
        upper(col("Severity")).isin("LOW", "MEDIUM", "HIGH", "CRITICAL")
    )
    .withColumn("severity_normalized", upper(col("Severity")))
    .withColumn("alert_hour", hour(col("CreatedAt")))
    .withColumn("alert_date", to_date(col("CreatedAt")))
    .withColumn(
        "alert_age_minutes",
        (
            unix_timestamp(current_timestamp()) -
            unix_timestamp(col("CreatedAt"))
        ) / 60
    )
    .withColumn(
        "risk_score",
        when(col("severity_normalized") == "LOW", 1)
        .when(col("severity_normalized") == "MEDIUM", 2)
        .when(col("severity_normalized") == "HIGH", 3)
        .otherwise(5)
    )
)

In [0]:
tourist_df = (
    spark.read.format("parquet")
    .load(f"{bronze_base}/tourist")
    .select(
        "TouristId",
        "AgencyId",
        "Nationality"
    )
    .dropDuplicates(["TouristId"])
)

agency_df = (
    spark.read.format("parquet")
    .load(f"{bronze_base}/agency")
    .select(
        "AgencyId",
        col("AgencyName").alias("agency_name")
    )
    .dropDuplicates(["AgencyId"])
)

danger_df = (
    spark.read.format("parquet")
    .load(f"{bronze_base}/danger_place")
    .select(
        col("DangerPlaceId"),
        col("Name").alias("danger_place_name"),
        col("Severity").alias("danger_place_severity"),
        "RadiusMeters"
    )
    .dropDuplicates(["DangerPlaceId"])
)

In [0]:
enriched_df = (
    clean_df
    .join(tourist_df, on="TouristId", how="left")
    .join(agency_df, on="AgencyId", how="left")
    .join(
        danger_df,
        clean_df.PlaceId == danger_df.DangerPlaceId,
        how="left"
    )
    .drop("DangerPlaceId")
    .dropDuplicates(["AlertId"])
)

In [0]:
from delta.tables import DeltaTable
def upsert_geofence(batch_df, batch_id):
    silver_path = f"{silver_base}/geofence/geofence_alert_clean"
    batch_df = batch_df.dropDuplicates(["AlertId"])
    if not DeltaTable.isDeltaTable(spark, silver_path):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
        )
    else:
        delta_table = DeltaTable.forPath(
            spark,
            silver_path
        )
        (
            delta_table.alias("target")
            .merge(
                batch_df.alias("source"),
                "target.AlertId = source.AlertId"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
silver_query = (
    enriched_df.writeStream
    .foreachBatch(upsert_geofence)
    .option(
        "checkpointLocation",
        f"{checkpoint_base}/silver"
    )
    .trigger(availableNow=True)
    .start()
)
silver_query.awaitTermination()

In [0]:
silver_df = spark.read.format("delta").load(
    f"{silver_base}/geofence/geofence_alert_clean"
)
display(silver_df)
print("Silver count:", silver_df.count())
print("Distinct AlertIds:", silver_df.select("AlertId").distinct().count())